# Session 7 — Loss Functions in Machine Learning

This notebook practices four common loss functions and then applies them to a practical machine-learning scenario:

- Mean Squared Error (MSE)
- Mean Absolute Error (MAE)
- Binary Cross-Entropy (BCE)
- Categorical Cross-Entropy

The examples use different datasets, naming, code organization, and presentation.

## 1. Mean Squared Error (MSE)

MSE measures the average squared difference between predicted and target values. Squaring makes larger errors contribute more strongly.

In [1]:
import numpy as np

def mse_loss(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    errors = actual - predicted
    return np.mean(errors ** 2)

actual_temperature = [22.0, 25.5, 19.0, 31.0, 27.5]
estimated_temperature = [21.0, 26.5, 20.0, 29.0, 28.0]

mse_value = mse_loss(actual_temperature, estimated_temperature)

print("Actual values     :", actual_temperature)
print("Predicted values  :", estimated_temperature)
print("MSE               :", round(mse_value, 4))

Actual values     : [22.0, 25.5, 19.0, 31.0, 27.5]
Predicted values  : [21.0, 26.5, 20.0, 29.0, 28.0]
MSE               : 1.45


**Observation:** because the errors are squared, a large prediction error has a stronger effect on MSE than a small error.

## 2. Mean Absolute Error (MAE)

MAE uses the absolute size of each prediction error, making it easier to interpret in the same units as the target.

In [2]:
def mae_loss(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return np.mean(np.abs(actual - predicted))

actual_delivery_days = [2, 4, 3, 7, 5, 2]
predicted_delivery_days = [3, 5, 2, 6, 5, 4]

mae_value = mae_loss(actual_delivery_days, predicted_delivery_days)

print("Actual delivery days    :", actual_delivery_days)
print("Predicted delivery days :", predicted_delivery_days)
print("MAE                     :", round(mae_value, 4))

Actual delivery days    : [2, 4, 3, 7, 5, 2]
Predicted delivery days : [3, 5, 2, 6, 5, 4]
MAE                     : 1.0


**Interpretation:** the MAE tells us the average absolute prediction error. Since this example measures days, the result is also expressed in days.

## 3. Binary Cross-Entropy (BCE)

BCE is commonly used when the target has two classes, such as yes/no or positive/negative.

In [3]:
def binary_log_loss(labels, probabilities):
    labels = np.asarray(labels, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)

    # Avoid taking log(0).
    probabilities = np.clip(probabilities, 1e-12, 1 - 1e-12)

    loss_terms = (
        labels * np.log(probabilities)
        + (1 - labels) * np.log(1 - probabilities)
    )

    return -np.mean(loss_terms)

clicked = [1, 0, 1, 1, 0, 0]
click_probability = [0.82, 0.25, 0.65, 0.91, 0.35, 0.12]

bce_value = binary_log_loss(clicked, click_probability)

print("Observed labels :", clicked)
print("Predicted P(yes):", click_probability)
print("BCE loss        :", round(bce_value, 4))

Observed labels : [1, 0, 1, 1, 0, 0]
Predicted P(yes): [0.82, 0.25, 0.65, 0.91, 0.35, 0.12]
BCE loss        : 0.2616


A probability close to the correct label receives a smaller penalty. A confident prediction in the wrong direction receives a much larger penalty.

## 4. Categorical Cross-Entropy

For a multi-class prediction, the loss evaluates the probability assigned to the correct class.

In [4]:
def categorical_log_loss(correct_class, probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    safe_probability = np.clip(probabilities[correct_class], 1e-12, 1 - 1e-12)
    return -np.log(safe_probability)

# Classes: ['red', 'green', 'blue', 'yellow']
class_names = ["red", "green", "blue", "yellow"]

true_class = 2
predicted_distribution = [0.08, 0.17, 0.60, 0.15]

ce_value = categorical_log_loss(true_class, predicted_distribution)

print("Classes:", class_names)
print("Correct class:", class_names[true_class])
print("Predicted probabilities:", predicted_distribution)
print("Categorical cross-entropy:", round(ce_value, 4))
print("Probability total:", sum(predicted_distribution))

Classes: ['red', 'green', 'blue', 'yellow']
Correct class: blue
Predicted probabilities: [0.08, 0.17, 0.6, 0.15]
Categorical cross-entropy: 0.5108
Probability total: 1.0


**Observation:** the correct class is `blue`, and the model assigns it probability `0.60`. If the model assigned a much smaller probability to the correct class, the cross-entropy loss would increase.

## 5. Quick Comparison

The following experiment shows how MSE and MAE react to one unusually large error.

In [5]:
normal_actual = np.array([10, 12, 14, 16, 18], dtype=float)
normal_prediction = np.array([11, 11, 15, 15, 19], dtype=float)

with_large_error = normal_prediction.copy()
with_large_error[2] = 24

print("Without large error")
print("  MSE:", round(mse_loss(normal_actual, normal_prediction), 3))
print("  MAE:", round(mae_loss(normal_actual, normal_prediction), 3))

print("\nWith one large error")
print("  MSE:", round(mse_loss(normal_actual, with_large_error), 3))
print("  MAE:", round(mae_loss(normal_actual, with_large_error), 3))

Without large error
  MSE: 1.0
  MAE: 1.0

With one large error
  MSE: 20.8
  MAE: 2.8


**Takeaway:** MSE reacts more strongly to a large outlier because the error is squared. MAE grows linearly with the size of the error.

## 6. Real-World Scenario: Online Course Recommendation

Imagine an online learning platform recommends courses to users.

The model predicts whether a learner will **enroll** in a recommended course.

- Target `1` → learner enrolled
- Target `0` → learner did not enroll
- Model output → probability of enrollment

### Suitable loss: Binary Cross-Entropy

This is a **binary classification** task because there are two possible outcomes. BCE is appropriate because it evaluates predicted probabilities rather than only the final 0/1 decision.

For example, predicting `0.95` when the actual result is `0` should be penalized more heavily than predicting `0.55` for the same negative example.

### Summary

| Task | Suitable loss |
|---|---|
| Predict a continuous numerical value | MSE / MAE |
| Binary yes/no prediction | Binary Cross-Entropy |
| One class among several classes | Categorical Cross-Entropy |

The best loss depends on the type of prediction and the behavior we want the model to optimize.